# Assignment 4 — Hyperparameter Optimization: Concepts & Interpretation

**Dataset:** UCI Wine Quality (Red) — 1,599 samples, 11 physicochemical features, binary quality target  
**Model:** Gradient Boosting Classifier (sklearn)  
**Methods:** Baseline → Grid Search → Random Search → Bayesian Optimization → Optuna (TPE, RandomSearch, HyperBand) → Population Based Training (Ray Tune)


## 0. Setup & Data Loading

In [ ]:
# Install required packages (run once)
import subprocess, sys
for pkg in ['scikit-optimize', 'optuna', 'ray[tune]', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('All packages ready')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import time

# Load dataset
df = pd.read_csv('/Users/yuangzou/Downloads/winequality-red.csv', sep=';')
print('Shape:', df.shape)
print('Quality distribution:')
print(df['quality'].value_counts().sort_index())
df.head()

In [ ]:
# Binarize target: quality >= 6 → Good (1), else Bad (0)
df['quality_binary'] = (df['quality'] >= 6).astype(int)
print('Binary target distribution:')
print(df['quality_binary'].value_counts())
print(f'Class balance: {df["quality_binary"].mean():.2%} positive')

X = df.drop(columns=['quality', 'quality_binary'])
y = df['quality_binary']

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Results tracker
results = {}

---
## 1. Baseline — Default Hyperparameters

We first train a GradientBoostingClassifier with **sklearn defaults** to establish a performance baseline.  
This is the reference point against which all tuned models will be compared.

In [ ]:
start = time.time()
baseline_model = GradientBoostingClassifier(random_state=42)
baseline_cv = cross_val_score(baseline_model, X_train, y_train, cv=cv, scoring='accuracy')
baseline_model.fit(X_train, y_train)
baseline_test = accuracy_score(y_test, baseline_model.predict(X_test))
elapsed = time.time() - start

results['Baseline'] = {
    'cv_mean': baseline_cv.mean(),
    'cv_std':  baseline_cv.std(),
    'test_acc': baseline_test,
    'time': elapsed,
    'best_params': baseline_model.get_params()
}

print('=== BASELINE (Default Hyperparameters) ===')
print(f'CV Accuracy:   {baseline_cv.mean():.4f} ± {baseline_cv.std():.4f}')
print(f'Test Accuracy: {baseline_test:.4f}')
print(f'Time:          {elapsed:.1f}s')
print(f'Default params: n_estimators={baseline_model.n_estimators}, '
      f'max_depth={baseline_model.max_depth}, '
      f'learning_rate={baseline_model.learning_rate}')

---
## 2. Grid Search

**How it works:** Exhaustively evaluates every combination of the specified hyperparameter values.  
**Pro:** Guaranteed to find the best combination within the defined grid.  
**Con:** Computationally expensive — scales exponentially with grid size (curse of dimensionality).

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':  [50, 100, 200],
    'max_depth':     [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
}
# 3 x 3 x 3 = 27 combinations x 5-fold CV = 135 fits

start = time.time()
gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
)
gs.fit(X_train, y_train)
elapsed = time.time() - start

gs_test = accuracy_score(y_test, gs.predict(X_test))
results['Grid Search'] = {
    'cv_mean':    gs.best_score_,
    'cv_std':     gs.cv_results_['std_test_score'][gs.best_index_],
    'test_acc':   gs_test,
    'time':       elapsed,
    'best_params': gs.best_params_
}

print('=== GRID SEARCH ===')
print(f'Best params:   {gs.best_params_}')
print(f'CV Accuracy:   {gs.best_score_:.4f} ± {results["Grid Search"]["cv_std"]:.4f}')
print(f'Test Accuracy: {gs_test:.4f}')
print(f'Time:          {elapsed:.1f}s')
print(f'Combinations evaluated: {len(gs.cv_results_["params"])}')

---
## 3. Random Search

**How it works:** Randomly samples hyperparameter combinations from specified distributions.  
**Pro:** Much faster than Grid Search; often finds equally good solutions with far fewer evaluations.  
**Key insight:** Random Search is surprisingly effective because not all hyperparameters matter equally — random sampling hits important dimensions more efficiently than a grid.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators':  randint(50, 300),
    'max_depth':     randint(2, 8),
    'learning_rate': uniform(0.01, 0.3),
    'subsample':     uniform(0.6, 0.4),
    'min_samples_split': randint(2, 20),
}

start = time.time()
rs = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_dist, n_iter=50, cv=cv, scoring='accuracy',
    random_state=42, n_jobs=-1, verbose=0
)
rs.fit(X_train, y_train)
elapsed = time.time() - start

rs_test = accuracy_score(y_test, rs.predict(X_test))
results['Random Search'] = {
    'cv_mean':    rs.best_score_,
    'cv_std':     rs.cv_results_['std_test_score'][rs.best_index_],
    'test_acc':   rs_test,
    'time':       elapsed,
    'best_params': rs.best_params_
}

print('=== RANDOM SEARCH ===')
print(f'Best params:   {rs.best_params_}')
print(f'CV Accuracy:   {rs.best_score_:.4f} ± {results["Random Search"]["cv_std"]:.4f}')
print(f'Test Accuracy: {rs_test:.4f}')
print(f'Time:          {elapsed:.1f}s')
print(f'Iterations sampled: 50 (vs 27 grid combinations, but wider search space)')

---
## 4. Bayesian Optimization (scikit-optimize)

**How it works:** Builds a probabilistic surrogate model (Gaussian Process) of the objective function.  
At each step it uses an **acquisition function** to decide where to sample next, balancing exploration vs exploitation.  
**Pro:** Much more sample-efficient than Grid or Random Search — learns from past evaluations.  
**Con:** Sequential by nature (each trial informs the next), harder to parallelize.

In [ ]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

bayes_space = {
    'n_estimators':      Integer(50, 300),
    'max_depth':         Integer(2, 8),
    'learning_rate':     Real(0.01, 0.3, prior='log-uniform'),
    'subsample':         Real(0.6, 1.0),
    'min_samples_split': Integer(2, 20),
}

start = time.time()
bayes = BayesSearchCV(
    GradientBoostingClassifier(random_state=42),
    bayes_space, n_iter=30, cv=cv, scoring='accuracy',
    random_state=42, n_jobs=-1, verbose=0
)
bayes.fit(X_train, y_train)
elapsed = time.time() - start

bayes_test = accuracy_score(y_test, bayes.predict(X_test))
results['Bayesian Opt.'] = {
    'cv_mean':    bayes.best_score_,
    'cv_std':     bayes.cv_results_['std_test_score'][bayes.best_index_],
    'test_acc':   bayes_test,
    'time':       elapsed,
    'best_params': dict(bayes.best_params_)
}

print('=== BAYESIAN OPTIMIZATION ===')
print(f'Best params:   {dict(bayes.best_params_)}')
print(f'CV Accuracy:   {bayes.best_score_:.4f} ± {results["Bayesian Opt."]["cv_std"]:.4f}')
print(f'Test Accuracy: {bayes_test:.4f}')
print(f'Time:          {elapsed:.1f}s')
print(f'Iterations: 30 (informed by surrogate model — more efficient than random)')

---
## 5. Optuna — TPE (Tree-structured Parzen Estimator)

**How it works:** Optuna's default sampler. Models the distribution of good vs bad hyperparameter configurations separately using Parzen estimators, then samples from regions likely to improve the objective.  
**Pro:** Highly efficient, handles conditional search spaces, built-in pruning support.  
**TPE insight:** Unlike Bayesian Optimization (which models p(y|x)), TPE models p(x|y) — it directly learns which hyperparameter values lead to good outcomes.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_tpe(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300),
        'max_depth':         trial.suggest_int('max_depth', 2, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state': 42
    }
    model = GradientBoostingClassifier(**params)
    score = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy').mean()
    return score

start = time.time()
study_tpe = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_tpe.optimize(objective_tpe, n_trials=50, show_progress_bar=False)
elapsed = time.time() - start

# Evaluate best model
best_tpe = GradientBoostingClassifier(**study_tpe.best_params, random_state=42)
tpe_cv = cross_val_score(best_tpe, X_train, y_train, cv=cv, scoring='accuracy')
best_tpe.fit(X_train, y_train)
tpe_test = accuracy_score(y_test, best_tpe.predict(X_test))

results['Optuna TPE'] = {
    'cv_mean':    tpe_cv.mean(),
    'cv_std':     tpe_cv.std(),
    'test_acc':   tpe_test,
    'time':       elapsed,
    'best_params': study_tpe.best_params
}

print('=== OPTUNA TPE ===')
print(f'Best params:   {study_tpe.best_params}')
print(f'Best trial CV: {study_tpe.best_value:.4f}')
print(f'CV Accuracy:   {tpe_cv.mean():.4f} ± {tpe_cv.std():.4f}')
print(f'Test Accuracy: {tpe_test:.4f}')
print(f'Time:          {elapsed:.1f}s (50 trials)')

---
## 6. Optuna — Random Search Sampler

**How it works:** Uses Optuna's framework but with pure random sampling (no learning between trials).  
This serves as a **controlled comparison** — same infrastructure as TPE, but without intelligent guidance.  
**Purpose:** Isolates the benefit of TPE's surrogate model over pure random search within Optuna.

In [ ]:
start = time.time()
study_rs = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.RandomSampler(seed=42))
study_rs.optimize(objective_tpe, n_trials=50, show_progress_bar=False)
elapsed = time.time() - start

best_rs_model = GradientBoostingClassifier(**study_rs.best_params, random_state=42)
rs_opt_cv = cross_val_score(best_rs_model, X_train, y_train, cv=cv, scoring='accuracy')
best_rs_model.fit(X_train, y_train)
rs_opt_test = accuracy_score(y_test, best_rs_model.predict(X_test))

results['Optuna Random'] = {
    'cv_mean':    rs_opt_cv.mean(),
    'cv_std':     rs_opt_cv.std(),
    'test_acc':   rs_opt_test,
    'time':       elapsed,
    'best_params': study_rs.best_params
}

print('=== OPTUNA RANDOM SEARCH ===')
print(f'Best params:   {study_rs.best_params}')
print(f'Best trial CV: {study_rs.best_value:.4f}')
print(f'CV Accuracy:   {rs_opt_cv.mean():.4f} ± {rs_opt_cv.std():.4f}')
print(f'Test Accuracy: {rs_opt_test:.4f}')
print(f'Time:          {elapsed:.1f}s (50 trials)')

---
## 7. Optuna — HyperBand Pruner

**How it works:** HyperBand is a **multi-fidelity** method. It runs many trials with fewer resources (e.g., fewer estimators/CV folds) and aggressively **prunes** poor-performing trials early — allocating more budget only to promising ones.  
**Pro:** Dramatically reduces computation by stopping bad trials early.  
**Con:** Requires an intermediate metric; may prune trials that start slowly but converge well ("slow starters").

In [ ]:
def objective_hyperband(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300),
        'max_depth':         trial.suggest_int('max_depth', 2, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state': 42
    }
    model = GradientBoostingClassifier(**params)
    # Report intermediate scores for pruning
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    for step, score in enumerate(cv_scores):
        trial.report(score, step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return cv_scores.mean()

start = time.time()
study_hb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner()
)
study_hb.optimize(objective_hyperband, n_trials=60, show_progress_bar=False)
elapsed = time.time() - start

pruned = sum(1 for t in study_hb.trials if t.state == optuna.trial.TrialState.PRUNED)
completed = sum(1 for t in study_hb.trials if t.state == optuna.trial.TrialState.COMPLETE)

best_hb_model = GradientBoostingClassifier(**study_hb.best_params, random_state=42)
hb_cv = cross_val_score(best_hb_model, X_train, y_train, cv=cv, scoring='accuracy')
best_hb_model.fit(X_train, y_train)
hb_test = accuracy_score(y_test, best_hb_model.predict(X_test))

results['Optuna HyperBand'] = {
    'cv_mean':    hb_cv.mean(),
    'cv_std':     hb_cv.std(),
    'test_acc':   hb_test,
    'time':       elapsed,
    'best_params': study_hb.best_params
}

print('=== OPTUNA HYPERBAND ===')
print(f'Best params:   {study_hb.best_params}')
print(f'CV Accuracy:   {hb_cv.mean():.4f} ± {hb_cv.std():.4f}')
print(f'Test Accuracy: {hb_test:.4f}')
print(f'Time:          {elapsed:.1f}s')
print(f'Trials: {completed} completed, {pruned} pruned (efficiency gain from early stopping)')

---
## 8. Population Based Training (PBT) with Ray Tune

**How it works:** PBT maintains a **population of models** training simultaneously. At regular intervals, poorly-performing models "exploit" better ones by copying their weights/params, then "explore" by randomly perturbing those params.  
**Pro:** Adapts hyperparameters *during* training (schedule), not just at the start — useful for models with training dynamics.  
**Con:** Overkill for static sklearn models; shines most with deep learning where hyperparameters can be annealed over epochs.

In [ ]:
import ray
from ray import tune
from ray.tune.schedulers import PopulationBasedTraining

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, log_to_driver=False, logging_level='error')

def train_gbc(config):
    model = GradientBoostingClassifier(
        n_estimators=int(config['n_estimators']),
        max_depth=int(config['max_depth']),
        learning_rate=config['learning_rate'],
        subsample=config['subsample'],
        random_state=42
    )
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy')
    tune.report(mean_accuracy=scores.mean())

pbt_scheduler = PopulationBasedTraining(
    time_attr='training_iteration',
    perturbation_interval=2,
    hyperparam_mutations={
        'learning_rate': tune.loguniform(0.01, 0.3),
        'subsample':     tune.uniform(0.6, 1.0),
        'n_estimators':  [50, 100, 150, 200, 250],
        'max_depth':     [2, 3, 4, 5, 6, 7],
    }
)

start = time.time()
analysis = tune.run(
    train_gbc,
    config={
        'n_estimators':  tune.choice([50, 100, 150, 200]),
        'max_depth':     tune.choice([3, 4, 5]),
        'learning_rate': tune.loguniform(0.01, 0.3),
        'subsample':     tune.uniform(0.6, 1.0),
    },
    num_samples=8,
    scheduler=pbt_scheduler,
    metric='mean_accuracy',
    mode='max',
    verbose=0,
    stop={'training_iteration': 5}
)
elapsed = time.time() - start
ray.shutdown()

best_pbt_config = analysis.best_config
best_pbt_model = GradientBoostingClassifier(
    n_estimators=int(best_pbt_config['n_estimators']),
    max_depth=int(best_pbt_config['max_depth']),
    learning_rate=best_pbt_config['learning_rate'],
    subsample=best_pbt_config['subsample'],
    random_state=42
)
pbt_cv = cross_val_score(best_pbt_model, X_train, y_train, cv=cv, scoring='accuracy')
best_pbt_model.fit(X_train, y_train)
pbt_test = accuracy_score(y_test, best_pbt_model.predict(X_test))

results['PBT (Ray Tune)'] = {
    'cv_mean':    pbt_cv.mean(),
    'cv_std':     pbt_cv.std(),
    'test_acc':   pbt_test,
    'time':       elapsed,
    'best_params': best_pbt_config
}

print('=== POPULATION BASED TRAINING (Ray Tune) ===')
print(f'Best config:   {best_pbt_config}')
print(f'CV Accuracy:   {pbt_cv.mean():.4f} ± {pbt_cv.std():.4f}')
print(f'Test Accuracy: {pbt_test:.4f}')
print(f'Time:          {elapsed:.1f}s')

---
## 9. Comprehensive Comparison & Visualization

In [ ]:
# Summary table
summary = pd.DataFrame({
    method: {
        'CV Accuracy (mean)': f"{v['cv_mean']:.4f}",
        'CV Accuracy (±std)': f"{v['cv_std']:.4f}",
        'Test Accuracy':      f"{v['test_acc']:.4f}",
        'Time (s)':           f"{v['time']:.1f}"
    }
    for method, v in results.items()
}).T

print('=== FULL RESULTS SUMMARY ===')
print(summary.to_string())
print()
best_method = max(results, key=lambda m: results[m]['test_acc'])
print(f'Best method by test accuracy: {best_method} ({results[best_method]["test_acc"]:.4f})')

In [ ]:
methods   = list(results.keys())
cv_means  = [results[m]['cv_mean']  for m in methods]
cv_stds   = [results[m]['cv_std']   for m in methods]
test_accs = [results[m]['test_acc'] for m in methods]
times     = [results[m]['time']     for m in methods]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Hyperparameter Optimization — Method Comparison\nUCI Wine Quality (Red)', 
             fontsize=14, fontweight='bold')

colors = plt.cm.Set2(np.linspace(0, 1, len(methods)))

# Plot 1: CV Accuracy with error bars
bars = axes[0].bar(methods, cv_means, yerr=cv_stds, capsize=5,
                   color=colors, edgecolor='black', linewidth=0.8)
axes[0].axhline(cv_means[0], color='red', linestyle='--', linewidth=1.5, label='Baseline')
axes[0].set_title('CV Accuracy (mean ± std)', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([min(cv_means)-0.05, 1.0])
axes[0].set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
axes[0].legend(fontsize=9)
for bar, val in zip(bars, cv_means):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# Plot 2: Test Accuracy
bars2 = axes[1].bar(methods, test_accs, color=colors, edgecolor='black', linewidth=0.8)
axes[1].axhline(test_accs[0], color='red', linestyle='--', linewidth=1.5, label='Baseline')
axes[1].set_title('Test Set Accuracy', fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([min(test_accs)-0.05, 1.0])
axes[1].set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
axes[1].legend(fontsize=9)
for bar, val in zip(bars2, test_accs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# Plot 3: Computation Time
bars3 = axes[2].bar(methods, times, color=colors, edgecolor='black', linewidth=0.8)
axes[2].set_title('Computation Time (seconds)', fontweight='bold')
axes[2].set_ylabel('Time (s)')
axes[2].set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
for bar, val in zip(bars3, times):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.0f}s', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('/Users/yuangzou/Downloads/hyperparameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# Efficiency plot: Accuracy vs Time
fig, ax = plt.subplots(figsize=(9, 6))
for i, method in enumerate(methods):
    ax.scatter(times[i], test_accs[i], s=200, color=colors[i], 
               label=method, zorder=5, edgecolors='black', linewidth=0.8)
    ax.annotate(method, (times[i], test_accs[i]),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Computation Time (s)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Efficiency Trade-off: Accuracy vs Computation Time', 
             fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('/Users/yuangzou/Downloads/efficiency_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Interpretation & Recommended Methodology

### Method-by-Method Analysis

**Baseline (Default Parameters)**  
Establishes the floor. sklearn defaults are reasonable starting points but are not dataset-specific. Any method that cannot beat the baseline raises questions about search space design, not the method itself.

**Grid Search**  
Exhaustive and deterministic. With a 3×3×3 grid (27 combinations × 5 folds = 135 fits), it guarantees finding the best combination *within the grid* — but the grid itself is a human-defined constraint. Misses the continuous space between grid points. Best used when the number of hyperparameters is small and their ranges are well-understood.

**Random Search**  
Searches a continuous, wider space with far fewer evaluations. Bergstra & Bengio (2012) showed that Random Search outperforms Grid Search when few hyperparameters dominate — which is the typical case. In practice it finds comparably good solutions to Grid Search with 5–10× fewer evaluations.

**Bayesian Optimization**  
The most principled sequential approach. By fitting a Gaussian Process surrogate, it avoids wasting evaluations in poor regions. With only 30 iterations it typically matches or exceeds Random Search at 50 iterations. The downside is that the GP surrogate is expensive to fit in high dimensions (>20 parameters).

**Optuna TPE**  
Scales better than Bayesian Optimization to high-dimensional spaces. TPE's key advantage is modeling p(x|y<y*) vs p(x|y≥y*) — it learns to distinguish "good" from "bad" parameter regions. Optuna's pruning integration and trial management make it the most practical tool for production ML pipelines.

**Optuna Random Search**  
The control group for Optuna TPE. Same framework, same search space, no learning between trials. Performance gap between Optuna Random and Optuna TPE directly quantifies the value of the surrogate model.

**Optuna HyperBand**  
HyperBand shines when individual evaluations are expensive and can be interrupted (e.g., deep learning with epochs). For sklearn GBC, each trial is a complete cross-validation run — pruning mid-way doesn't fully exploit HyperBand's strength. Pruned trials did save some compute, but "slow starters" may have been unfairly eliminated.

**Population Based Training (Ray Tune)**  
PBT is architecturally designed for iterative training processes (neural networks with gradient descent). Its exploit/explore mechanism adapts hyperparameters *during* training. For a static sklearn model (hyperparameters set once before fitting), PBT offers limited benefit over Optuna TPE. The overhead of Ray's distributed framework adds latency without proportional gain.

---

### Recommended Methodology: **Optuna with TPE**

For this specific task (GradientBoostingClassifier on a tabular dataset), **Optuna TPE** is the recommended approach for these reasons:

1. **Best accuracy-to-compute ratio:** TPE reaches competitive performance faster than Grid/Random Search by intelligently exploring the space.

2. **Continuous search space:** Unlike Grid Search which is constrained to a discrete lattice, Optuna samples from continuous distributions — exploring the full hyperparameter landscape.

3. **Scalability:** Handles high-dimensional spaces better than Gaussian Process-based Bayesian Optimization, which becomes computationally expensive beyond ~20 parameters.

4. **Practical tooling:** Optuna's API is clean, supports pruning, parallel execution, visualization, and study persistence — making it production-ready.

5. **When optimization becomes a liability:** Over-optimizing on CV score risks **overfitting the validation set** — especially with small datasets like this (1,599 samples). The gap between CV and test accuracy should be monitored. If CV accuracy improves but test accuracy stagnates, the search has overfit the CV splits.

**Final recommendation:** Use Optuna TPE with 50–100 trials, monitor the CV/test gap, and stop early if test performance plateaus — this balances thoroughness with generalization.

In [ ]:
# Optuna TPE optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial values over time
trial_values = [t.value for t in study_tpe.trials if t.value is not None]
best_so_far = [max(trial_values[:i+1]) for i in range(len(trial_values))]

axes[0].scatter(range(len(trial_values)), trial_values, alpha=0.5, s=30, color='steelblue', label='Trial score')
axes[0].plot(range(len(best_so_far)), best_so_far, color='red', linewidth=2, label='Best so far')
axes[0].set_title('Optuna TPE — Optimization History', fontweight='bold')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('CV Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Parameter importance (n_estimators vs learning_rate)
lr_vals  = [t.params['learning_rate'] for t in study_tpe.trials if t.value is not None]
acc_vals = [t.value for t in study_tpe.trials if t.value is not None]
sc = axes[1].scatter(lr_vals, acc_vals, c=acc_vals, cmap='RdYlGn', s=60, alpha=0.8)
plt.colorbar(sc, ax=axes[1], label='CV Accuracy')
axes[1].set_title('Learning Rate vs CV Accuracy (Optuna TPE)', fontweight='bold')
axes[1].set_xlabel('learning_rate (log scale)')
axes[1].set_ylabel('CV Accuracy')
axes[1].set_xscale('log')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/yuangzou/Downloads/optuna_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Optuna analysis plot saved.')

In [ ]:
# Final best model classification report
from sklearn.metrics import classification_report, confusion_matrix

print('=== FINAL MODEL: Optuna TPE Best Configuration ===')
print(f'Best params: {study_tpe.best_params}')
print()
print(classification_report(y_test, best_tpe.predict(X_test),
                             target_names=['Low Quality (0)', 'High Quality (1)']))

cm = confusion_matrix(y_test, best_tpe.predict(X_test))
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Low', 'Pred High'],
            yticklabels=['True Low', 'True High'])
ax.set_title('Confusion Matrix — Optuna TPE Best Model', fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/yuangzou/Downloads/confusion_matrix.png', dpi=150)
plt.show()